# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## Setup & Imports

In [3]:
!nvidia-smi # Take a look at the GPU

Thu Mar 26 20:12:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   23C    P8              1W /  250W |    1884MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install numpy
%pip install torch torchvision
%pip install wandb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from typing import List, Dict, Tuple, Any

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make results reproducible
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Data Loading

In [5]:
BATCH_SIZE = 256 # Same batch size throughout notebook

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

classes: list[str] = train_set.classes
print("Classes:", classes)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [6]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## Training & Evaluation Helpers

`train_one_epoch` and `validate` each handle a single pass through their respective loader.  
`fit` orchestrates the loop, tracks history, and restores the best checkpoint.

In [7]:
def train(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> Tuple[float, float]:
    """Run one full pass over `loader` in training mode."""
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct      += (outputs.argmax(dim=1) == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def validate(
    model: nn.Module,
    loader: DataLoader,
) -> Tuple[float, float]:
    """Run one full pass over `loader` in evaluation mode."""
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            correct      += (outputs.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    wandb_kwargs: Dict[str, Any],
    num_epochs: int = 10,
) -> Dict[str, List[float]]:
    """Train `model` for `num_epochs`, validating after every epoch.

    Initialises and closes a wandb run for the duration of training.
    Saves the weights that achieved the best validation accuracy and
    restores them at the end.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """

    with wandb.init(**wandb_kwargs):

        best_val_acc = 0.0
        best_state   = None
        history: dict[str, list[float]] = {
            "train_loss": [], "val_loss": [],
            "train_acc":  [], "val_acc":  [],
        }
    
        for epoch in range(1, num_epochs + 1):
            train_loss, train_acc = train(model, train_loader, optimizer)
            val_loss,   val_acc   = validate(model, val_loader)
    
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
    
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
            print(
                f"Epoch {epoch:>{len(str(num_epochs))}}/{num_epochs} | "
                f"train loss {train_loss:.4f}, train acc {train_acc:.2f}% | "
                f"val loss {val_loss:.4f}, val acc {val_acc:.2f}%"
            )
    
            wandb.log({
                "Training Loss": train_loss,
                "Training Accuracy": train_acc,
                "Validation Loss": val_loss,
                "Validation Accuracy": val_acc
            })
    
    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best weights (val acc {best_val_acc:.2f}%)")

    return history

## Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [8]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-2

criterion = nn.CrossEntropyLoss()

model_a = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="SGD + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "SGD", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_a = fit(model_a, optimizer_a, train_loader, test_loader,
                wandb_kwargs=wandb_kwargs, num_epochs=NUM_EPOCHS)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: hamid-sabeti (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch  1/50 | train loss 2.2997, train acc 12.46% | val loss 2.2955, val acc 14.96%
Epoch  2/50 | train loss 2.2896, train acc 16.31% | val loss 2.2804, val acc 19.16%
Epoch  3/50 | train loss 2.2573, train acc 21.53% | val loss 2.2130, val acc 24.55%
Epoch  4/50 | train loss 2.1299, train acc 25.05% | val loss 2.0493, val acc 27.11%
Epoch  5/50 | train loss 2.0100, train acc 28.21% | val loss 1.9597, val acc 30.06%
Epoch  6/50 | train loss 1.9273, train acc 31.04% | val loss 1.8898, val acc 32.39%
Epoch  7/50 | train loss 1.8482, train acc 34.13% | val loss 1.8185, val acc 35.67%
Epoch  8/50 | train loss 1.7711, train acc 36.90% | val loss 1.7491, val acc 36.58%
Epoch  9/50 | train loss 1.7061, train acc 38.81% | val loss 1.7711, val acc 36.83%
Epoch 10/50 | train loss 1.6457, train acc 41.00% | val loss 1.6359, val acc 40.28%
Epoch 11/50 | train loss 1.6030, train acc 42.19% | val loss 1.5628, val acc 43.98%
Epoch 12/50 | train loss 1.5626, train acc 43.60% | val loss 1.5525, val acc

Training Accuracy,▁▁▂▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇██████
Training Loss,███▇▇▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
Validation Accuracy,▁▂▂▃▃▄▄▄▅▅▅▅▆▅▆▆▆▆▆▇▆▇▆▆▇▇▇▇▇▇▇█▆███▇███
Validation Loss,███▇▆▅▅▅▄▄▄▄▃▄▃▃▃▂▃▃▃▃▂▂▂▂▂▂▂▂▃▁▂▂▁▂▁▁▁▁
Training Accuracy,69.304
Training Loss,0.88161
Validation Accuracy,60.75
Validation Loss,1.11922



Restored best weights (val acc 64.32%)


## Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [9]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_b = fit(model_b, optimizer_b, train_loader, test_loader,
                wandb_kwargs=wandb_kwargs, num_epochs=NUM_EPOCHS)

Epoch  1/50 | train loss 1.5439, train acc 43.91% | val loss 1.2988, val acc 52.73%
Epoch  2/50 | train loss 1.1452, train acc 59.38% | val loss 1.0524, val acc 62.13%
Epoch  3/50 | train loss 0.9338, train acc 67.30% | val loss 0.8916, val acc 68.66%
Epoch  4/50 | train loss 0.8015, train acc 72.04% | val loss 0.8409, val acc 70.40%
Epoch  5/50 | train loss 0.7024, train acc 75.61% | val loss 0.8251, val acc 72.27%
Epoch  6/50 | train loss 0.6171, train acc 78.66% | val loss 0.7879, val acc 72.76%
Epoch  7/50 | train loss 0.5428, train acc 81.09% | val loss 0.7649, val acc 74.42%
Epoch  8/50 | train loss 0.4702, train acc 83.68% | val loss 0.7836, val acc 73.94%
Epoch  9/50 | train loss 0.4043, train acc 85.92% | val loss 0.7647, val acc 75.12%
Epoch 10/50 | train loss 0.3357, train acc 88.39% | val loss 0.7492, val acc 76.60%
Epoch 11/50 | train loss 0.2708, train acc 90.75% | val loss 0.8016, val acc 76.58%
Epoch 12/50 | train loss 0.2182, train acc 92.74% | val loss 0.8868, val acc

Training Accuracy,▁▃▄▅▅▆▆▇▇▇██████████████████████████████
Training Loss,█▆▅▅▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▄▆▆▇▇▇███▇██▇████████▇███▇████████▇████
Validation Loss,▄▃▂▁▁▁▁▁▁▁▂▃▃▃▄▄▄▅▄▅▆▆▆▆▆▇▇▇▇▇▇▇▇█▇█████
Training Accuracy,99.19
Training Loss,0.02476
Validation Accuracy,75.57
Validation Loss,2.04457



Restored best weights (val acc 76.60%)


## Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [10]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + Tanh",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "Tanh",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_c = fit(model_c, optimizer_c, train_loader, test_loader,
                wandb_kwargs=wandb_kwargs, num_epochs=NUM_EPOCHS)

Epoch  1/50 | train loss 1.4087, train acc 49.39% | val loss 1.1576, val acc 58.92%
Epoch  2/50 | train loss 1.0227, train acc 64.04% | val loss 0.9811, val acc 65.31%
Epoch  3/50 | train loss 0.8467, train acc 70.47% | val loss 0.9014, val acc 68.59%
Epoch  4/50 | train loss 0.7271, train acc 74.94% | val loss 0.8095, val acc 71.86%
Epoch  5/50 | train loss 0.6165, train acc 78.83% | val loss 0.7742, val acc 73.09%
Epoch  6/50 | train loss 0.5233, train acc 82.24% | val loss 0.7644, val acc 73.83%
Epoch  7/50 | train loss 0.4385, train acc 85.49% | val loss 0.7618, val acc 74.48%
Epoch  8/50 | train loss 0.3532, train acc 88.85% | val loss 0.7907, val acc 74.34%
Epoch  9/50 | train loss 0.2765, train acc 91.52% | val loss 0.7761, val acc 74.89%
Epoch 10/50 | train loss 0.2055, train acc 94.28% | val loss 0.8272, val acc 74.35%
Epoch 11/50 | train loss 0.1504, train acc 96.40% | val loss 0.8499, val acc 74.46%
Epoch 12/50 | train loss 0.0993, train acc 98.15% | val loss 0.8680, val acc

Training Accuracy,▁▃▄▅▅▆▆▇▇███████████████████████████████
Training Loss,█▆▅▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▄▆▇▇███████████████████████████████████
Validation Loss,▅▃▂▁▁▁▁▁▂▂▂▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇████
Training Accuracy,100
Training Loss,0.00011
Validation Accuracy,75.3
Validation Loss,1.49584



Restored best weights (val acc 75.46%)
